In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Any
import sys
sys.path.insert(0, "D:\\STUDY\\Programming\\project\\house-price-prediction")

from src.data_loader import DataLoader

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("✅ Libraries imported")

✅ Libraries imported


In [2]:
loader = DataLoader("../config/config.yaml")
df = loader.load_data()
profile = loader.profile_data(df)

print(f"Shape: {df.shape}")
df.head()


INFO:src.data_loader:✅ Config loaded from: ../config/config.yaml
INFO:src.data_loader:✅ DataLoader initialized successfully.
INFO:src.data_loader:🔄 Loading data from: D:\STUDY\Programming\project\house-price-prediction\data\raw\housing_data.csv
INFO:src.data_loader:✅ Data loaded successfully. Shape: (90000, 28)
INFO:src.data_loader:🔄 Generating data profile...
INFO:src.data_loader:✅ Data profiling complete.



          📊 DATA PROFILE SUMMARY
  Total Rows      : 90,000
  Total Columns   : 28
  Numerical Cols  : 14
  Categorical Cols: 14
  Missing Values  : 5420
  Duplicates      : 0
  Memory Usage    : 82.85 MB
-------------------------------------------------------
  Target Column   : SalePrice
  Mean Price      : $298,493.11
  Median Price    : $288,000.00
  Price Range     : $67,500 - $755,000
  Skewness        : 0.7856
Shape: (90000, 28)


,LotArea,YearBuilt,YearRemodAdd,TotalBsmtSF,GrLivArea,FullBath,HalfBath,BedroomAbvGr,TotRmsAbvGrd,GarageArea,PoolArea,OverallQual,OverallCond,MSZoning,Street,LotShape,LandContour,Utilities,Neighborhood,BldgType,HouseStyle,RoofStyle,ExterQual,HeatingQC,KitchenQual,GarageType,SaleCondition,SalePrice
0,11051,1901,1901,982,1447,0,0,2,4,466,0,6,8,RL,Pave,Reg,Low,AllPub,NAmes,1Fam,1Story,Gable,Gd,Ex,TA,BuiltIn,Partial,233400
1,6930,1918,1918,648,1449,2,0,3,9,536,422,9,8,RL,Pave,Reg,Lvl,AllPub,OldTown,1Fam,2Story,Gable,Gd,Gd,TA,Detchd,Partial,312900
2,7588,1906,2009,0,1376,0,0,3,7,0,0,8,5,RL,Pave,Reg,Lvl,AllPub,OldTown,1Fam,2Story,Hip,TA,TA,TA,NaN,Normal,210800
3,10887,1894,1894,781,1327,1,2,3,8,424,0,7,5,C (all),Pave,IR1,Lvl,AllPub,Edwards,1Fam,1Story,Gable,TA,Ex,TA,CarPort,AdjLand,196600
4,10573,1915,1915,891,1124,0,0,1,5,345,309,5,5,FV,Pave,Reg,Lvl,AllPub,Mitchel,1Fam,1Story,Gable,TA,Ex,TA,CarPort,Normal,217700


In [3]:
# Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original distribution
axes[0].hist(df["SalePrice"], bins=50, color="#2563EB", edgecolor="white")
axes[0].set_title("Sale Price Distribution (Original)")
axes[0].set_xlabel("Sale Price ($)")
axes[0].set_ylabel("Frequency")

Text(0, 0.5, 'Frequency')

In [4]:
# Log-transformed distribution
import numpy as np
axes[1].hist(np.log1p(df["SalePrice"]), bins=50, 
             color="#10B981", edgecolor="white")
axes[1].set_title("Sale Price Distribution (Log Transformed)")
axes[1].set_xlabel("Log(Sale Price)")

plt.suptitle("Target Variable Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/target_distribution.png", dpi=150)
plt.show()

print(f"Skewness (original):    {df['SalePrice'].skew():.4f}")
print(f"Skewness (log-transform): {np.log1p(df['SalePrice']).skew():.4f}")


Skewness (original):    0.7856
Skewness (log-transform): -0.1773


In [5]:
# Correlation Analysis
num_df = df.select_dtypes(include=[np.number])
corr = num_df.corr()["SalePrice"].sort_values(ascending=False)

print("Top 10 correlated features with SalePrice:")
print(corr.head(11))

Top 10 correlated features with SalePrice:
SalePrice      1.00
GrLivArea      0.55
OverallQual    0.49
TotalBsmtSF    0.42
FullBath       0.27
TotRmsAbvGrd   0.25
GarageArea     0.24
BedroomAbvGr   0.22
YearBuilt      0.09
PoolArea       0.09
YearRemodAdd   0.09
Name: SalePrice, dtype: float64


In [6]:
# Correlation Heatmap
top_features = corr.head(11).index.tolist()
plt.figure(figsize=(12, 10))
sns.heatmap(
    df[top_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    square=True,
)
plt.title("Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/correlation_heatmap.png", dpi=150)
plt.show()

In [7]:
# Key Feature Relationships 
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
features = ["GrLivArea", "OverallQual", "GarageArea",
            "TotalBsmtSF", "YearBuilt", "FullBath"]

for i, feat in enumerate(features):
    ax = axes[i // 3][i % 3]
    ax.scatter(df[feat], df["SalePrice"], alpha=0.4, color="#2563EB", s=20)
    ax.set_xlabel(feat)
    ax.set_ylabel("Sale Price ($)")
    ax.set_title(f"{feat} vs Sale Price")

plt.suptitle("Feature vs Sale Price Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [8]:
# Missing Value Analysis
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

print("Missing Value Summary:")
print(pd.DataFrame({"Count": missing, "Percentage": missing_pct}))

Missing Value Summary:
            Count  Percentage
GarageType   5420        6.02


In [9]:
# Categorical Feature Analysis
cat_cols = df.select_dtypes(include=["object"]).columns[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, col in enumerate(cat_cols):
    ax = axes[i // 2][i % 2]
    order = df.groupby(col)["SalePrice"].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y="SalePrice", order=order, ax=ax)
    ax.set_title(f"{col} vs Sale Price")
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()